# Objectives
I aim to compare the effectiveness of traditional forecasting models like Naive, Seasonal Naive, ARIMA, SARIMA, and SARIMAX and machine learning models like Decision Tree, Random Forest, XGBoost, and Neural Network. The comparison is implemented on sales data of Favorita stores located in Ecuador. The training data includes dates, store and product information, whether that item was being promoted, as well as the sales numbers. Additional files include supplementary information that may be useful in building your models.

# File Descriptions and Data Field Information

**train.csv**

- The training data, comprising time series of features `store_nbr`, `family`, and `onpromotion` as well as the target sales.
- `store_nbr` identifies the store at which the products are sold.
- `family` identifies the type of product sold.
- `sales` gives the total sales for a product family at a particular store at a given date. Fractional values are possible since products can be sold in fractional units (1.5 kg of cheese, for instance, as opposed to 1 bag of chips).
- `onpromotion` gives the total number of items in a product family that were being promoted at a store at a given date.

----------------------------------------------

**test.csv**

- The test data, having the same features as the training data.
- The dates in the test data are for the 15 days after the last date in the training data.

----------------------------------------------

**stores.csv**

- Store metadata, including `city`, `state`, `type`, and `cluster`.
- `cluster` is a grouping of similar stores.

----------------------------------------------

**oil.csv**

- Daily oil price. Includes values during both the train and test data timeframes. (Ecuador is an oil-dependent country and it's economical health is highly vulnerable to shocks in oil prices.)

----------------------------------------------

**holidays_events.csv**

- Holidays and Events, with metadata
- **NOTE:** Pay special attention to the `transferred` column. A holiday that is `transferred` officially falls on that calendar day, but was moved to another date by the government. A `transferred` day is more like a normal day than a holiday. To find the day that it was actually celebrated, look for the corresponding row where type is Transfer. For example, the holiday Independencia de Guayaquil was `transferred` from 2012-10-09 to 2012-10-12, which means it was celebrated on 2012-10-12. Days that are type Bridge are extra days that are added to a holiday (e.g., to extend the break across a long weekend). These are frequently made up by the type Work Day which is a day not normally scheduled for work (e.g., Saturday) that is meant to payback the Bridge.
- Additional holidays are days added a regular calendar holiday, for example, as typically happens around Christmas (making Christmas Eve a holiday).

**Additional Notes**

- Wages in the public sector are paid every two weeks on the 15th and on the last day of the month. Demand could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Citation
Alexis Cook, DanB, inversion, and Ryan Holbrook. Store Sales - Time Series Forecasting. https://kaggle.com/competitions/store-sales-time-series-forecasting, 2021. Kaggle.

In [1]:
import os
import numpy as np # mathematic calculations
import pandas as pd # exploratory data analysis
import scipy.stats as stats # statistics
import matplotlib.pyplot as plt # data visualization
import seaborn as sns # data visualization
import datetime as dt # date data manipulation
import time # training and predicting time recording
import ast
import joblib

# Import Time Series Analysis libraries
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, month_plot, quarter_plot
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.api import ARIMA, SARIMAX

# Libraries for Machine Learning Models
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_log_error
import category_encoders as ce

# Neural Network
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
from keras.models import Model
import keras_tuner as kt
from scikeras.wrappers import KerasRegressor
from scipy.stats import loguniform, randint, uniform

In [83]:
MAIN = '/Users/nguyendinhphong/Documents/Supply Chain Analysis/projects/Inventory Optimization Modeling/Demand Forecasting System'
RESULT = '/Users/nguyendinhphong/Documents/Supply Chain Analysis/projects/Inventory Optimization Modeling/Demand Forecasting System/results/'
MODELS = '/Users/nguyendinhphong/Documents/Supply Chain Analysis/projects/Inventory Optimization Modeling/Demand Forecasting System/models/'
DATA = '/Users/nguyendinhphong/Documents/Supply Chain Analysis/projects/Inventory Optimization Modeling/Demand Forecasting System/data/'

In [3]:
# Get train and test data
train_df = pd.read_csv(os.path.join(MAIN, 'train.csv'))
test_df = pd.read_csv(os.path.join(MAIN, 'test.csv'))
train_df.info()
train_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [4]:
train_df['store_nbr'].value_counts()

store_nbr
1     55572
46    55572
36    55572
37    55572
38    55572
39    55572
4     55572
40    55572
41    55572
42    55572
43    55572
44    55572
45    55572
47    55572
10    55572
48    55572
49    55572
5     55572
50    55572
51    55572
52    55572
53    55572
54    55572
6     55572
7     55572
8     55572
35    55572
34    55572
33    55572
32    55572
11    55572
12    55572
13    55572
14    55572
15    55572
16    55572
17    55572
18    55572
19    55572
2     55572
20    55572
21    55572
22    55572
23    55572
24    55572
25    55572
26    55572
27    55572
28    55572
29    55572
3     55572
30    55572
31    55572
9     55572
Name: count, dtype: int64

In [5]:
holiday_df = pd.read_csv(os.path.join(MAIN, 'holidays_events.csv'))
oil_df = pd.read_csv(os.path.join(MAIN, 'full_oil.csv'))
oil_df['date'] = pd.to_datetime(oil_df['date'], format='%m/%d/%Y')
transaction_df = pd.read_csv(os.path.join(MAIN, 'transactions.csv'))
store_df = pd.read_csv(os.path.join(MAIN, 'stores.csv'))

In [6]:
# Check info of each dataframe
train_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           28512 non-null  int64 
 1   date         28512 non-null  object
 2   store_nbr    28512 non-null  int64 
 3   family       28512 non-null  object
 4   onpromotion  28512 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 1.1+ MB


In [7]:
holiday_df.info()
oil_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9957 entries, 0 to 9956
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    9957 non-null   datetime64[ns]
 1   price   9957 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 155.7 KB


In [8]:
transaction_df.info()
store_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


In [9]:
# Convert date column to datetime format
train_df['date'] = pd.to_datetime(train_df['date']).dt.date
test_df['date'] = pd.to_datetime(test_df['date']).dt.date
holiday_df['date'] = pd.to_datetime(holiday_df['date']).dt.date
oil_df['date'] = pd.to_datetime(oil_df['date']).dt.date
transaction_df['date'] = pd.to_datetime(transaction_df['date']).dt.date

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])
holiday_df['date'] = pd.to_datetime(holiday_df['date'])
oil_df['date'] = pd.to_datetime(oil_df['date'])
transaction_df['date'] = pd.to_datetime(transaction_df['date'])

# Convert `store_nbr` and `family` to categorical data type
train_df['store_nbr'] = train_df['store_nbr'].astype('category')
test_df['store_nbr'] = test_df['store_nbr'].astype('category')
train_df['family'] = train_df['family'].astype('category')
test_df['family'] = test_df['family'].astype('category')

# Convert `id` to string type
train_df['id'] = train_df['id'].astype(str)
test_df['id'] = test_df['id'].astype(str)

In [10]:
train_df.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [11]:
train_df.date.describe()

count                          3000888
mean     2015-04-24 08:27:04.703088384
min                2013-01-01 00:00:00
25%                2014-02-26 18:00:00
50%                2015-04-24 12:00:00
75%                2016-06-19 06:00:00
max                2017-08-15 00:00:00
Name: date, dtype: object

# Checking missing data

In [12]:
train_df.isnull().sum(), test_df.isnull().sum()

(id             0
 date           0
 store_nbr      0
 family         0
 sales          0
 onpromotion    0
 dtype: int64,
 id             0
 date           0
 store_nbr      0
 family         0
 onpromotion    0
 dtype: int64)

# Processing

In [13]:
# Rename type in holiday_df to holiday_type
holiday_df.rename(columns={'type': 'holiday_type'}, inplace=True)

# Create new features for holiday_df
holiday_types = ['Holiday', 'Transfer', 'Additional', 'Bridge', 'Event']
holiday_df['is_holiday'] = np.where((holiday_df['holiday_type'].isin(holiday_types)) & (holiday_df['transferred'] == False), 1, 0)
holiday_df['is_bridge_day'] = np.where(holiday_df['holiday_type'] == 'Bridge', 1, 0)
holiday_df['is_local_holiday'] = np.where((holiday_df['locale'] == 'Local') & (holiday_df['is_holiday'] == 1), 1, 0)

In [14]:
# Author said there was an earthquake in 2016-04-16 and it impacted the sales in several weeks
# I assume that "several weeks" means 4 weeks before and after the earthquake
earthquake_date = pd.to_datetime('2016-04-16').date()
end_of_effect_date = earthquake_date + pd.Timedelta(weeks=4)

mask = (holiday_df['date'].dt.date >= earthquake_date) & (holiday_df['date'].dt.date <= end_of_effect_date)
holiday_df['days_since_earthquake'] = 0

# For the rows where the date is within the earthquake effect period (in mask), add 1 to the number of days since earthquake
# ensure 'date' is datetime64 then add one day to masked rows
holiday_df['date'] = pd.to_datetime(holiday_df['date'])
holiday_df.loc[mask, 'days_since_earthquake'] = (holiday_df.loc[mask, 'date'] - pd.Timestamp(earthquake_date)).dt.days + 1

In [15]:
holiday_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   350 non-null    datetime64[ns]
 1   holiday_type           350 non-null    object        
 2   locale                 350 non-null    object        
 3   locale_name            350 non-null    object        
 4   description            350 non-null    object        
 5   transferred            350 non-null    bool          
 6   is_holiday             350 non-null    int64         
 7   is_bridge_day          350 non-null    int64         
 8   is_local_holiday       350 non-null    int64         
 9   days_since_earthquake  350 non-null    int64         
dtypes: bool(1), datetime64[ns](1), int64(4), object(4)
memory usage: 25.1+ KB


In [16]:
holiday_df.days_since_earthquake.value_counts()

days_since_earthquake
0     316
16      2
27      2
6       2
23      2
22      2
28      1
26      1
25      1
24      1
21      1
20      1
19      1
18      1
17      1
15      1
1       1
14      1
13      1
12      1
11      1
10      1
9       1
8       1
7       1
5       1
4       1
3       1
2       1
29      1
Name: count, dtype: int64

In [17]:
holiday_df[['date', 'days_since_earthquake']][holiday_df['days_since_earthquake'] > 0]

,date,days_since_earthquake
219,2016-04-16,1
220,2016-04-17,2
221,2016-04-18,3
222,2016-04-19,4
223,2016-04-20,5
224,2016-04-21,6
225,2016-04-21,6
226,2016-04-22,7
227,2016-04-23,8
228,2016-04-24,9


In [18]:
holiday_df.date.describe()

count                              350
mean     2015-04-24 00:45:15.428571392
min                2012-03-02 00:00:00
25%                2013-12-23 06:00:00
50%                2015-06-08 00:00:00
75%                2016-07-03 00:00:00
max                2017-12-26 00:00:00
Name: date, dtype: object

In [19]:
oil_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9957 entries, 0 to 9956
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    9957 non-null   datetime64[ns]
 1   price   9957 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 155.7 KB


In [20]:
oil_df.date.describe()

count                             9957
mean     2005-09-18 09:14:54.847845632
min                1986-01-02 00:00:00
25%                1995-10-13 00:00:00
50%                2005-09-19 00:00:00
75%                2015-08-10 00:00:00
max                2025-07-21 00:00:00
Name: date, dtype: object

In [21]:
oil_df

,date,price
0,2025-07-21,68.39
1,2025-07-18,68.53
2,2025-07-17,68.76
3,2025-07-16,67.13
4,2025-07-15,67.76
...,...,...
9952,1986-01-08,25.87
9953,1986-01-07,25.85
9954,1986-01-06,26.53
9955,1986-01-03,26.00


In [22]:
# check the value date of index 9950 of oil_df
oil_df.loc[9950].date

Timestamp('1986-01-10 00:00:00')

In [23]:
oil_df["date"] = (
    oil_df["date"]
    .astype(str)
    .str.replace(" ", "")
)

oil_df["date"] = pd.to_datetime(oil_df["date"], errors="coerce")

oil_df = oil_df.sort_values("date").drop_duplicates("date")

oil_df["price"] = oil_df["price"].ffill()

In [24]:
# Convert `is_holiday`, `is_bridge_day`, `is_local_holiday` to string type
holiday_df['is_holiday'] = holiday_df['is_holiday'].astype(str)
holiday_df['is_bridge_day'] = holiday_df['is_bridge_day'].astype(str)
holiday_df['is_local_holiday'] = holiday_df['is_local_holiday'].astype(str)

# Convert `transferred` column to boolean type
holiday_df['transferred'] = holiday_df['transferred'].astype(bool)

# Convert `type` column to categorical type
holiday_df['holiday_type'] = holiday_df['holiday_type'].astype('category')

# Keep date part of the datetime
holiday_df['date'] = holiday_df['date'].dt.date
oil_df['date'] = oil_df['date'].dt.date

holiday_df['date'] = pd.to_datetime(holiday_df['date'])
oil_df['date'] = pd.to_datetime(oil_df['date'])

# Rename price in oil_df to oil_price
oil_df.rename(columns={'price': 'oil_price'}, inplace=True)

In [25]:
holiday_df

,date,holiday_type,locale,locale_name,description,transferred,is_holiday,is_bridge_day,is_local_holiday,days_since_earthquake
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False,1,0,1,0
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False,1,0,0,0
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False,1,0,1,0
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False,1,0,1,0
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False,1,0,0,0
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False,1,0,0,0
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False,1,0,0,0
348,2017-12-25,Holiday,National,Ecuador,Navidad,False,1,0,0,0


In [26]:
# Convert `store_nbr` to categorical type
transaction_df['store_nbr'] = transaction_df['store_nbr'].astype('category')
store_df['store_nbr'] = store_df['store_nbr'].astype('category')

# Convert `cluster` to categorical type
store_df['cluster'] = store_df['cluster'].astype('category')

# Build time series datasets

In [27]:
# Merge train and test datasets with holiday, oil, transaction, and store data
train_merge_1 = pd.merge(train_df, holiday_df, on='date', how='left')
train_merge_2 = pd.merge(train_merge_1, store_df, on='store_nbr', how='left')
train_merge_3 = pd.merge(train_merge_2, oil_df, on='date', how='left')

# Merge test dataset with holiday, oil, transaction, and store data
test_merge_1 = pd.merge(test_df, holiday_df, on='date', how='left')
test_merge_2 = pd.merge(test_merge_1, store_df, on='store_nbr', how='left')
test_merge_3 = pd.merge(test_merge_2, oil_df, on='date', how='left')

final_train_df = train_merge_3.copy()
final_test_df = test_merge_3.copy()

In [28]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3054348 entries, 0 to 3054347
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id                     object        
 1   date                   datetime64[ns]
 2   store_nbr              category      
 3   family                 category      
 4   sales                  float64       
 5   onpromotion            int64         
 6   holiday_type           category      
 7   locale                 object        
 8   locale_name            object        
 9   description            object        
 10  transferred            object        
 11  is_holiday             object        
 12  is_bridge_day          object        
 13  is_local_holiday       object        
 14  days_since_earthquake  float64       
 15  city                   object        
 16  state                  object        
 17  type                   object        
 18  cluster               

In [29]:
final_train_df

,id,date,store_nbr,family,sales,onpromotion,holiday_type,locale,locale_name,description,transferred,is_holiday,is_bridge_day,is_local_holiday,days_since_earthquake,city,state,type,cluster,oil_price
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0,Holiday,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13,NaN
1,1,2013-01-01,1,BABY CARE,0.000,0,Holiday,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13,NaN
2,2,2013-01-01,1,BEAUTY,0.000,0,Holiday,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13,NaN
3,3,2013-01-01,1,BEVERAGES,0.000,0,Holiday,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13,NaN
4,4,2013-01-01,1,BOOKS,0.000,0,Holiday,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3054343,3000883,2017-08-15,9,POULTRY,438.133,0,Holiday,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,B,6,47.57
3054344,3000884,2017-08-15,9,PREPARED FOODS,154.553,1,Holiday,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,B,6,47.57
3054345,3000885,2017-08-15,9,PRODUCE,2419.729,148,Holiday,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,B,6,47.57
3054346,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8,Holiday,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,B,6,47.57


In [30]:
final_train_df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion',
       'holiday_type', 'locale', 'locale_name', 'description', 'transferred',
       'is_holiday', 'is_bridge_day', 'is_local_holiday',
       'days_since_earthquake', 'city', 'state', 'type', 'cluster',
       'oil_price'],
      dtype='object')

In [31]:
agg_rules = {
    'sales': 'sum',
    'store_nbr': 'first',
    'onpromotion': 'sum',
    'holiday_type': 'first',
    'oil_price': 'first',
    'locale': 'first',
    'locale_name': 'first',
    'description': 'first',
    'transferred': 'first',
    'is_holiday': 'first',
    'is_bridge_day': 'first',
    'is_local_holiday': 'first',
    'days_since_earthquake': 'first',
    'city': 'first',
    'state': 'first',
    'type': 'first',
    'cluster': 'first'
}

# 1. Sort by date, family, and sales (sales in descending order)
final_train_df = final_train_df.sort_values(
    by=['date', 'family', 'sales'], 
    ascending=[True, True, False]
)

# 2. Run your exact same groupby and agg
final_train_df = final_train_df.groupby(['date', 'family']).agg(agg_rules).reset_index()

agg_rules = {
    'onpromotion': 'sum',
    'store_nbr': 'first',
    'holiday_type': 'first',
    'oil_price': 'first',
    'locale': 'first',
    'locale_name': 'first',
    'description': 'first',
    'transferred': 'first',
    'is_holiday': 'first',
    'is_bridge_day': 'first',
    'is_local_holiday': 'first',
    'days_since_earthquake': 'first',
    'city': 'first',
    'state': 'first',
    'type': 'first',
    'cluster': 'first'
}
# 1. Sort by date, family
final_test_df = final_test_df.sort_values(
    by=['date', 'family'], 
    ascending=[True, True]
)

# 2. Run your exact same groupby and agg
final_test_df = final_test_df.groupby(['date', 'family']).agg(agg_rules).reset_index()

/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/2307086962.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_train_df = final_train_df.groupby(['date', 'family']).agg(agg_rules).reset_index()
/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/2307086962.py:55: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_test_df = final_test_df.groupby(['date', 'family']).agg(agg_rules).reset_index()


In [32]:
final_train_df.duplicated().sum(), final_test_df.duplicated().sum()

(np.int64(0), np.int64(0))

In [33]:
final_train_df

,date,family,sales,store_nbr,onpromotion,holiday_type,oil_price,locale,locale_name,description,transferred,is_holiday,is_bridge_day,is_local_holiday,days_since_earthquake,city,state,type,cluster
0,2013-01-01,AUTOMOTIVE,0.000000,1,0,Holiday,NaN,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13
1,2013-01-01,BABY CARE,0.000000,1,0,Holiday,NaN,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13
2,2013-01-01,BEAUTY,2.000000,25,0,Holiday,NaN,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Salinas,Santa Elena,D,1
3,2013-01-01,BEVERAGES,810.000000,25,0,Holiday,NaN,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Salinas,Santa Elena,D,1
4,2013-01-01,BOOKS,0.000000,1,0,Holiday,NaN,National,Ecuador,Primer dia del ano,False,1,0,0,0.0,Quito,Pichincha,D,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55567,2017-08-15,POULTRY,17586.709986,3,6,Holiday,47.57,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,D,8
55568,2017-08-15,PREPARED FOODS,4641.522980,44,9,Holiday,47.57,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,A,5
55569,2017-08-15,PRODUCE,125108.971000,3,3169,Holiday,47.57,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,D,8
55570,2017-08-15,SCHOOL AND OFFICE SUPPLIES,2530.000000,45,148,Holiday,47.57,Local,Riobamba,Fundacion de Riobamba,False,1,0,1,0.0,Quito,Pichincha,A,11


In [34]:
# Set holiday_type to string
final_train_df['holiday_type'] = final_train_df['holiday_type'].astype(str)
final_test_df['holiday_type'] = final_test_df['holiday_type'].astype(str)

# To fill missing values in holiday_type
# 1. Extract day and month from date (not year)
holiday_df['month_day'] = holiday_df['date'].dt.strftime('%m-%d')
holidays = set(holiday_df[holiday_df['holiday_type'] == 'Holiday']['month_day'].values)

# 2. Create conditions
condition_holiday_1 = final_train_df['holiday_type'].isna()
condition_holiday_2 = final_train_df['holiday_type'] == 'nan'
condition_holiday_3 = final_train_df['date'].isin(holidays)

condition_weekend_1 = final_train_df['holiday_type'].isna()
condition_weekend_2 = final_train_df['holiday_type'] == 'nan'
condition_weekend_3 = final_train_df['date'].dt.dayofweek >= 5

# 3. Fill missing values for Holiday
final_train_df.loc[condition_holiday_1 & condition_holiday_2 & condition_holiday_3, 'holiday_type'] = 'Holiday'

# 4. Fill missing values for Weekend
final_train_df.loc[condition_weekend_1 & condition_weekend_2 & condition_weekend_3, 'holiday_type'] = 'Weekend'

# 5. Fill missing values for Work Day
final_train_df['holiday_type'].fillna('Work Day', inplace=True)
final_train_df['holiday_type'] = np.where(final_train_df['holiday_type'] == 'nan', 'Work Day', final_train_df['holiday_type'])

# -------------------------------------------------------------------------------
# Do the same for final_test_df
# 1. Extract day and month from date (not year)
# 2. Create conditions
condition_holiday_1 = final_test_df['holiday_type'].isna()
condition_holiday_2 = final_test_df['holiday_type'] == 'nan'
condition_holiday_3 = final_test_df['date'].isin(holidays)

condition_weekend_1 = final_test_df['holiday_type'].isna()
condition_weekend_2 = final_test_df['holiday_type'] == 'nan'
condition_weekend_3 = final_test_df['date'].dt.dayofweek >= 5

# 3. Fill missing values for Holiday
final_test_df.loc[condition_holiday_1 & condition_holiday_2 & condition_holiday_3, 'holiday_type'] = 'Holiday'

# 4. Fill missing values for Weekend
final_test_df.loc[condition_weekend_1 & condition_weekend_2 & condition_weekend_3, 'holiday_type'] = 'Weekend'

# 5. Fill missing values for Work Day
final_test_df['holiday_type'].fillna('Work Day', inplace=True)
final_test_df['holiday_type'] = np.where(final_test_df['holiday_type'] == 'nan', 'Work Day', final_test_df['holiday_type'])

/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/1903434756.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_train_df['holiday_type'].fillna('Work Day', inplace=True)
/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/1903434756.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are se

In [35]:
final_train_df['holiday_type'].isna().sum(), final_test_df['holiday_type'].isna().sum()

(np.int64(0), np.int64(0))

In [36]:
final_train_df['holiday_type'].value_counts()

holiday_type
Work Day      47388
Holiday        5016
Event          1617
Additional     1188
Transfer        264
Bridge           99
Name: count, dtype: int64

In [37]:
final_test_df['holiday_type'].value_counts()

holiday_type
Work Day    495
Holiday      33
Name: count, dtype: int64

# Filling missing value of oil_price of the final datasets

In [38]:
final_train_df.date.isna().sum(), final_test_df.date.isna().sum()

(np.int64(0), np.int64(0))

In [39]:
final_train_df.oil_price.isna().sum(), final_test_df.oil_price.isna().sum()

(np.int64(17193), np.int64(132))

In [40]:
empty_oil_date = final_train_df[final_train_df['oil_price'].isna()].date.unique()
empty_oil_date

<DatetimeArray>
['2013-01-01 00:00:00', '2013-01-05 00:00:00', '2013-01-06 00:00:00',
 '2013-01-12 00:00:00', '2013-01-13 00:00:00', '2013-01-19 00:00:00',
 '2013-01-20 00:00:00', '2013-01-21 00:00:00', '2013-01-26 00:00:00',
 '2013-01-27 00:00:00',
 ...
 '2017-07-15 00:00:00', '2017-07-16 00:00:00', '2017-07-22 00:00:00',
 '2017-07-23 00:00:00', '2017-07-29 00:00:00', '2017-07-30 00:00:00',
 '2017-08-05 00:00:00', '2017-08-06 00:00:00', '2017-08-12 00:00:00',
 '2017-08-13 00:00:00']
Length: 521, dtype: datetime64[ns]

In [41]:
oil_df

,date,oil_price
9956,1986-01-02,25.56
9955,1986-01-03,26.00
9954,1986-01-06,26.53
9953,1986-01-07,25.85
9952,1986-01-08,25.87
...,...,...
4,2025-07-15,67.76
3,2025-07-16,67.13
2,2025-07-17,68.76
1,2025-07-18,68.53


In [42]:
# check the index 2990 of oil_df
oil_df.loc[3142].date, oil_df.loc[3141].date

(Timestamp('2013-01-04 00:00:00'), Timestamp('2013-01-07 00:00:00'))

In [43]:
# fill missing oil_price with forward fill method
final_train_df['oil_price'] = final_train_df['oil_price'].ffill().bfill()
final_test_df['oil_price'] = final_test_df['oil_price'].ffill().bfill()

In [44]:
final_train_df.oil_price.isna().sum(), final_test_df.oil_price.isna().sum()

(np.int64(0), np.int64(0))

# Filling missing value of days_since_earthquake of final datasets

In [45]:
final_train_df[final_train_df.isna().any(axis=1)]

,date,family,sales,store_nbr,onpromotion,holiday_type,oil_price,locale,locale_name,description,transferred,is_holiday,is_bridge_day,is_local_holiday,days_since_earthquake,city,state,type,cluster
33,2013-01-02,AUTOMOTIVE,255.000000,44,0,Work Day,93.14,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,5
34,2013-01-02,BABY CARE,0.000000,1,0,Work Day,93.14,None,None,None,None,None,None,None,NaN,Quito,Pichincha,D,13
35,2013-01-02,BEAUTY,207.000000,46,0,Work Day,93.14,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,14
36,2013-01-02,BEVERAGES,72092.000000,44,0,Work Day,93.14,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,5
37,2013-01-02,BOOKS,0.000000,1,0,Work Day,93.14,None,None,None,None,None,None,None,NaN,Quito,Pichincha,D,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55534,2017-08-14,POULTRY,18597.508060,45,7,Work Day,47.59,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,11
55535,2017-08-14,PREPARED FOODS,4647.375002,44,8,Work Day,47.59,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,5
55536,2017-08-14,PRODUCE,115257.595980,44,325,Work Day,47.59,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,5
55537,2017-08-14,SCHOOL AND OFFICE SUPPLIES,2826.000000,45,154,Work Day,47.59,None,None,None,None,None,None,None,NaN,Quito,Pichincha,A,11


In [46]:
# Fill missing value of days_since_earthquake by 2 ways:
# 1. If the date is after the earthquake date, fill by adding 1 to "days_since_earthquake" of the previous day and keep going to the end of dataset
# 2. If the date is before the earthquake date, fill with 0
final_train_df['days_since_earthquake'] = np.where(final_train_df['date'] < pd.Timestamp(earthquake_date), 0, final_train_df['days_since_earthquake'])
final_train_df['days_since_earthquake'] = final_train_df['days_since_earthquake'].ffill().bfill()

# For test dataset, I will fill days_since_earthquake of the first day by the days_since_earthquake of the last day of the train dataset, then fill the rest by forward fill method
final_test_df.iloc[0, final_test_df.columns.get_loc('days_since_earthquake')] = final_train_df['days_since_earthquake'].iloc[-1]
final_test_df['days_since_earthquake'] = final_test_df['days_since_earthquake'].ffill().bfill()

In [47]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55572 entries, 0 to 55571
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   55572 non-null  datetime64[ns]
 1   family                 55572 non-null  category      
 2   sales                  55572 non-null  float64       
 3   store_nbr              55572 non-null  category      
 4   onpromotion            55572 non-null  int64         
 5   holiday_type           55572 non-null  object        
 6   oil_price              55572 non-null  float64       
 7   locale                 8316 non-null   object        
 8   locale_name            8316 non-null   object        
 9   description            8316 non-null   object        
 10  transferred            8316 non-null   object        
 11  is_holiday             8316 non-null   object        
 12  is_bridge_day          8316 non-null   object        
 13  i

In [48]:
final_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   528 non-null    datetime64[ns]
 1   family                 528 non-null    category      
 2   onpromotion            528 non-null    int64         
 3   store_nbr              528 non-null    category      
 4   holiday_type           528 non-null    object        
 5   oil_price              528 non-null    float64       
 6   locale                 33 non-null     object        
 7   locale_name            33 non-null     object        
 8   description            33 non-null     object        
 9   transferred            33 non-null     object        
 10  is_holiday             33 non-null     object        
 11  is_bridge_day          33 non-null     object        
 12  is_local_holiday       33 non-null     object        
 13  days_

**NOTE 04-15-2026:** Dealing with missing values before convertin to weekly demand.

In [49]:
final_train_df['is_holiday'].value_counts()

is_holiday
1    7887
0     429
Name: count, dtype: int64

In [50]:
final_train_df['is_holiday'].values

array(['1', '1', '1', ..., '1', '1', '1'], shape=(55572,), dtype=object)

# Filling missing values of is_holiday by taking dates of these days and encoding these dates that have missing values of is_holiday

In [51]:
# 1. Identify all unique 'month-day' combinations that are known holidays
holiday_month_days = holiday_df[holiday_df['is_holiday'] == "1"]['date'].dt.strftime('%m-%d').unique()

def fill_missing_is_holiday(df, holiday_mds):
    # Extract the 'month-day' string for every date in the dataframe
    df_month_day = df['date'].dt.strftime('%m-%d')
    
    # Create a Series of "1"s and "0"s based on whether the month-day is in the known holiday list
    fill_values = pd.Series(np.where(df_month_day.isin(holiday_mds), "1", "0"), index=df.index)
    
    # Fill the missing values in 'is_holiday'
    df['is_holiday'] = df['is_holiday'].fillna(fill_values)
    return df

# 2. Apply the function to both train and test dataframes
final_train_df = fill_missing_is_holiday(final_train_df, holiday_month_days)
final_test_df = fill_missing_is_holiday(final_test_df, holiday_month_days)

# 3. Verify that the missing values have been filled
print("Missing 'is_holiday' in train:", final_train_df['is_holiday'].isna().sum())
print("Missing 'is_holiday' in test:", final_test_df['is_holiday'].isna().sum())


Missing 'is_holiday' in train: 0
Missing 'is_holiday' in test: 0


In [52]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55572 entries, 0 to 55571
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   55572 non-null  datetime64[ns]
 1   family                 55572 non-null  category      
 2   sales                  55572 non-null  float64       
 3   store_nbr              55572 non-null  category      
 4   onpromotion            55572 non-null  int64         
 5   holiday_type           55572 non-null  object        
 6   oil_price              55572 non-null  float64       
 7   locale                 8316 non-null   object        
 8   locale_name            8316 non-null   object        
 9   description            8316 non-null   object        
 10  transferred            8316 non-null   object        
 11  is_holiday             55572 non-null  object        
 12  is_bridge_day          8316 non-null   object        
 13  i

In [53]:
# 1. Identify all unique 'month-day' combinations that are known holidays
holiday_month_days = holiday_df[holiday_df['is_local_holiday'] == "1"]['date'].dt.strftime('%m-%d').unique()

def fill_missing_is_local_holiday(df, holiday_mds):
    # Extract the 'month-day' string for every date in the dataframe
    df_month_day = df['date'].dt.strftime('%m-%d')
    
    # Create a Series of "1"s and "0"s based on whether the month-day is in the known holiday list
    fill_values = pd.Series(np.where(df_month_day.isin(holiday_mds), "1", "0"), index=df.index)
    
    # Fill the missing values in 'is_local_holiday'
    df['is_local_holiday'] = df['is_local_holiday'].fillna(fill_values)
    return df

# 2. Apply the function to both train and test dataframes
final_train_df = fill_missing_is_local_holiday(final_train_df, holiday_month_days)
final_test_df = fill_missing_is_local_holiday(final_test_df, holiday_month_days)

# 3. Verify that the missing values have been filled
print("Missing 'is_local_holiday' in train:", final_train_df['is_local_holiday'].isna().sum())
print("Missing 'is_local_holiday' in test:", final_test_df['is_local_holiday'].isna().sum())


Missing 'is_local_holiday' in train: 0
Missing 'is_local_holiday' in test: 0


In [54]:
# 1. Identify all unique 'month-day' combinations that are known bridge days
holiday_month_days = holiday_df[holiday_df['is_bridge_day'] == "1"]['date'].dt.strftime('%m-%d').unique()

def fill_missing_is_bridge_day(df, holiday_mds):
    # Extract the 'month-day' string for every date in the dataframe
    df_month_day = df['date'].dt.strftime('%m-%d')
    
    # Create a Series of "1"s and "0"s based on whether the month-day is in the known holiday list
    fill_values = pd.Series(np.where(df_month_day.isin(holiday_mds), "1", "0"), index=df.index)
    
    # Fill the missing values in 'is_bridge_day'
    df['is_bridge_day'] = df['is_bridge_day'].fillna(fill_values)
    return df

# 2. Apply the function to both train and test dataframes
final_train_df = fill_missing_is_bridge_day(final_train_df, holiday_month_days)
final_test_df = fill_missing_is_bridge_day(final_test_df, holiday_month_days)

# 3. Verify that the missing values have been filled
print("Missing 'is_bridge_day' in train:", final_train_df['is_bridge_day'].isna().sum())
print("Missing 'is_bridge_day' in test:", final_test_df['is_bridge_day'].isna().sum())


Missing 'is_bridge_day' in train: 0
Missing 'is_bridge_day' in test: 0


In [55]:
holiday_df.holiday_type.value_counts()

holiday_type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

In [56]:
# 1. Create a mapping dictionary from 'month-day' to its corresponding 'holiday_type'
# I replace 'Transfer' with 'Work Day' right in the mapping so any filled values use 'Work Day'
md_to_holiday_type = (
    holiday_df.assign(month_day=holiday_df['date'].dt.strftime('%m-%d'))
    .drop_duplicates('month_day')
    .set_index('month_day')['holiday_type']
    .replace('Transfer', 'Work Day') 
    .to_dict()
)

def fill_missing_holiday_type(df, type_mapping):
    # Convert column to 'object' to avoid Categorical type errors
    df['holiday_type'] = df['holiday_type'].astype(object)
    
    # Extract the 'month-day' string
    df_md = df['date'].dt.strftime('%m-%d')
    
    # Fill missing values by mapping the extracted month-day to our dictionary
    df['holiday_type'] = df['holiday_type'].fillna(df_md.map(type_mapping))
    
    # For dates that do not belong to any type of holiday, fill with "Work Day"
    df['holiday_type'] = df['holiday_type'].fillna("Work Day")
    
    return df

# 2. Apply the adjusted function to both dataframes
final_train_df = fill_missing_holiday_type(final_train_df, md_to_holiday_type)
final_test_df = fill_missing_holiday_type(final_test_df, md_to_holiday_type)

# 3. Verify the values
print(final_train_df['holiday_type'].value_counts(dropna=False))

holiday_type
Work Day      47388
Holiday        5016
Event          1617
Additional     1188
Transfer        264
Bridge           99
Name: count, dtype: int64


/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/3754027744.py:7: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace('Transfer', 'Work Day')


In [57]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55572 entries, 0 to 55571
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   55572 non-null  datetime64[ns]
 1   family                 55572 non-null  category      
 2   sales                  55572 non-null  float64       
 3   store_nbr              55572 non-null  category      
 4   onpromotion            55572 non-null  int64         
 5   holiday_type           55572 non-null  object        
 6   oil_price              55572 non-null  float64       
 7   locale                 8316 non-null   object        
 8   locale_name            8316 non-null   object        
 9   description            8316 non-null   object        
 10  transferred            8316 non-null   object        
 11  is_holiday             55572 non-null  object        
 12  is_bridge_day          55572 non-null  object        
 13  i

In [58]:
# Fill any remaining missing values in 'is_local_holiday' with "0"
final_train_df['is_local_holiday'] = final_train_df['is_local_holiday'].fillna("0")
final_test_df['is_local_holiday'] = final_test_df['is_local_holiday'].fillna("0")

# Encode "1" to 1 and "0" to 0 by casting the data type to integer
cols_to_encode = ['is_holiday', 'is_local_holiday', 'is_bridge_day']

for col in cols_to_encode:
    final_train_df[col] = final_train_df[col].astype(bool)
    final_test_df[col] = final_test_df[col].astype(bool)

# Verify the data types to ensure they are now integers
print("Train data types:\n", final_train_df[cols_to_encode].dtypes)
print("\nTest data types:\n", final_test_df[cols_to_encode].dtypes)


Train data types:
 is_holiday          bool
is_local_holiday    bool
is_bridge_day       bool
dtype: object

Test data types:
 is_holiday          bool
is_local_holiday    bool
is_bridge_day       bool
dtype: object


In [59]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55572 entries, 0 to 55571
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   55572 non-null  datetime64[ns]
 1   family                 55572 non-null  category      
 2   sales                  55572 non-null  float64       
 3   store_nbr              55572 non-null  category      
 4   onpromotion            55572 non-null  int64         
 5   holiday_type           55572 non-null  object        
 6   oil_price              55572 non-null  float64       
 7   locale                 8316 non-null   object        
 8   locale_name            8316 non-null   object        
 9   description            8316 non-null   object        
 10  transferred            8316 non-null   object        
 11  is_holiday             55572 non-null  bool          
 12  is_bridge_day          55572 non-null  bool          
 13  i

In [60]:
final_train_df['days_since_earthquake'] = final_train_df['days_since_earthquake'].astype(int)
final_test_df['days_since_earthquake'] = final_test_df['days_since_earthquake'].astype(int)

In [61]:
def object_describe(df, col):
    display(df[col].describe())
    display(df[col].value_counts())
    print(f'The number of missing values: {df[col].isna().sum()}')
    print(f'The number of null values: {df[col].isnull().sum()}')

def num_describe(df, col):
    display(df[col].describe())
    print(f'The number of missing values: {df[col].isna().sum()}')
    print(f'The number of null values: {df[col].isnull().sum()}')

In [62]:
for col in final_train_df.select_dtypes(include=['object', 'category']).columns:
    print('================================')
    print(f"Column: {col}")
    object_describe(final_train_df, col)
    print("\n")

Column: family


count          55572
unique            33
top       AUTOMOTIVE
freq            1684
Name: family, dtype: object

family
AUTOMOTIVE                    1684
HOME APPLIANCES               1684
SCHOOL AND OFFICE SUPPLIES    1684
PRODUCE                       1684
PREPARED FOODS                1684
POULTRY                       1684
PLAYERS AND ELECTRONICS       1684
PET SUPPLIES                  1684
PERSONAL CARE                 1684
MEATS                         1684
MAGAZINES                     1684
LIQUOR,WINE,BEER              1684
LINGERIE                      1684
LAWN AND GARDEN               1684
LADIESWEAR                    1684
HOME CARE                     1684
HOME AND KITCHEN II           1684
BABY CARE                     1684
HOME AND KITCHEN I            1684
HARDWARE                      1684
GROCERY II                    1684
GROCERY I                     1684
FROZEN FOODS                  1684
EGGS                          1684
DELI                          1684
DAIRY                         1684
CLEANING                      1684
CELEBRATION                   1684
BREAD/BAKERY 

The number of missing values: 0
The number of null values: 0


Column: store_nbr


count     55572
unique       54
top          44
freq      18665
Name: store_nbr, dtype: int64

...

type
A    34622
D    16826
B     2356
C     1364
E      404
Name: count, dtype: int64

The number of missing values: 0
The number of null values: 0


Column: cluster


count     55572
unique       17
top           5
freq      18665
Name: cluster, dtype: int64

cluster
5     18665
13     8666
14     8641
11     6691
8      5912
6      2038
1      1033
10      664
17      625
15      624
3       569
4       500
2       376
9       276
16      121
12       91
7        80
Name: count, dtype: int64

The number of missing values: 0
The number of null values: 0




In [63]:
for col in final_train_df.select_dtypes(exclude=['object', 'category']).columns:
    print('================================')
    print(f"Column: {col}")
    num_describe(final_train_df, col)
    print("\n")

Column: date


count                            55572
mean     2015-04-24 08:27:04.703088128
min                2013-01-01 00:00:00
25%                2014-02-26 18:00:00
50%                2015-04-24 12:00:00
75%                2016-06-19 06:00:00
max                2017-08-15 00:00:00
Name: date, dtype: object

The number of missing values: 0
The number of null values: 0


Column: sales


count     55572.000000
mean      19732.504574
std       48280.015726
min           0.000000
25%          99.000000
50%        1077.000000
75%       15221.919254
max      952765.716000
Name: sales, dtype: float64

The number of missing values: 0
The number of null values: 0


Column: onpromotion


...

The number of missing values: 0
The number of null values: 0


Column: is_local_holiday


count     55572
unique        1
top        True
freq      55572
Name: is_local_holiday, dtype: object

The number of missing values: 0
The number of null values: 0


Column: days_since_earthquake


count    55572.000000
mean         0.258314
std          2.239091
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         29.000000
Name: days_since_earthquake, dtype: float64

The number of missing values: 0
The number of null values: 0




# Creating datasets

In [64]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55572 entries, 0 to 55571
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   55572 non-null  datetime64[ns]
 1   family                 55572 non-null  category      
 2   sales                  55572 non-null  float64       
 3   store_nbr              55572 non-null  category      
 4   onpromotion            55572 non-null  int64         
 5   holiday_type           55572 non-null  object        
 6   oil_price              55572 non-null  float64       
 7   locale                 8316 non-null   object        
 8   locale_name            8316 non-null   object        
 9   description            8316 non-null   object        
 10  transferred            8316 non-null   object        
 11  is_holiday             55572 non-null  bool          
 12  is_bridge_day          55572 non-null  bool          
 13  i

In [65]:
inventory_variables_train = final_train_df[['date', 'family', 'sales']]

In [ ]:
# Compute the weekly mean of sales, grouped by week ending date
inventory_df = inventory_variables_train.groupby([
    pd.Grouper(key='date', freq='W-SUN'), 
    'family'
])['sales'].mean().reset_index()

/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/3285555882.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  inventory_df = inventory_variables_train.groupby([


In [85]:
inventory_test_df = final_test_df[['date', 'family']]

In [75]:
inventory_df

,date,family,sales
0,2013-01-06,AUTOMOTIVE,214.500000
1,2013-01-06,BABY CARE,0.000000
2,2013-01-06,BEAUTY,153.833333
3,2013-01-06,BEVERAGES,57196.000000
4,2013-01-06,BOOKS,0.000000
...,...,...,...
7981,2017-08-20,POULTRY,18092.109023
7982,2017-08-20,PREPARED FOODS,4644.448991
7983,2017-08-20,PRODUCE,120183.283490
7984,2017-08-20,SCHOOL AND OFFICE SUPPLIES,2678.000000


In [71]:
forecast_df = final_train_df[['date', 'sales', 'onpromotion', 'holiday_type', 'is_holiday', 'is_bridge_day', 'days_since_earthquake', 'oil_price', 'family', 'store_nbr', 'cluster']]
forecast_df

,date,sales,onpromotion,holiday_type,is_holiday,is_bridge_day,days_since_earthquake,oil_price,family,store_nbr,cluster
0,2013-01-01,0.000000,0,Holiday,True,True,0,93.14,AUTOMOTIVE,1,13
1,2013-01-01,0.000000,0,Holiday,True,True,0,93.14,BABY CARE,1,13
2,2013-01-01,2.000000,0,Holiday,True,True,0,93.14,BEAUTY,25,1
3,2013-01-01,810.000000,0,Holiday,True,True,0,93.14,BEVERAGES,25,1
4,2013-01-01,0.000000,0,Holiday,True,True,0,93.14,BOOKS,1,13
...,...,...,...,...,...,...,...,...,...,...,...
55567,2017-08-15,17586.709986,6,Holiday,True,True,0,47.57,POULTRY,3,8
55568,2017-08-15,4641.522980,9,Holiday,True,True,0,47.57,PREPARED FOODS,44,5
55569,2017-08-15,125108.971000,3169,Holiday,True,True,0,47.57,PRODUCE,3,8
55570,2017-08-15,2530.000000,148,Holiday,True,True,0,47.57,SCHOOL AND OFFICE SUPPLIES,45,11


In [86]:
final_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   528 non-null    datetime64[ns]
 1   family                 528 non-null    category      
 2   onpromotion            528 non-null    int64         
 3   store_nbr              528 non-null    category      
 4   holiday_type           528 non-null    object        
 5   oil_price              528 non-null    float64       
 6   locale                 33 non-null     object        
 7   locale_name            33 non-null     object        
 8   description            33 non-null     object        
 9   transferred            33 non-null     object        
 10  is_holiday             528 non-null    bool          
 11  is_bridge_day          528 non-null    bool          
 12  is_local_holiday       528 non-null    bool          
 13  days_

In [ ]:
# ==========================================
# 1. WEEKLY AGGREGATIONS
# ==========================================

# A. Aggregate purely Date-Level features (1 row per day -> 1 row per week)
date_features = final_train_df[['date', 'is_holiday', 'days_since_earthquake', 'oil_price']].drop_duplicates()

weekly_date_features = date_features.groupby(pd.Grouper(key='date', freq='W-SUN')).agg(
    holiday_count=('is_holiday', 'sum'),
    is_holiday_week=('is_holiday', 'max'),
    max_days_since_earthquake=('days_since_earthquake', 'max'),
    avg_oil_price=('oil_price', 'mean')
).reset_index()

# B. Aggregate Sales and Promotion by Family and Week
weekly_sales = final_train_df.groupby(['family', pd.Grouper(key='date', freq='W-SUN')]).agg(
    sales=('sales', 'sum'),
    onpromotion=('onpromotion', 'sum')
).reset_index()

# C. Merge them together to get our base weekly dataframe
weekly_df = pd.merge(weekly_sales, weekly_date_features, on='date', how='left')


# ==========================================
# 2. DATE FEATURES
# ==========================================

# .ngroup() + 1 gives a continuous sequential week number for the entire time series (1, 2, 3...)
weekly_df['total_week_number'] = weekly_df.groupby('date').ngroup() + 1
weekly_df['month'] = weekly_df['date'].dt.month
weekly_df['quarter'] = weekly_df['date'].dt.quarter
weekly_df['year'] = weekly_df['date'].dt.year


# ==========================================
# 3. TIME-SERIES LAG & ROLLING FEATURES
# ==========================================

# Sort chronologically by family and date before creating time series features
weekly_df = weekly_df.sort_values(['family', 'date'])

# Group by 'family' so our lags and rolling means don't bleed from one product family into another
family_group = weekly_df.groupby('family')['sales']

# Create Lag Features
for lag in [1, 2, 3, 4, 12, 13]:
    weekly_df[f'lag_{lag}'] = family_group.shift(lag)

# Create Rolling Features
# IMPORTANT: I use .shift(1) BEFORE .rolling() to prevent data leakage. 
# This ensures I am not using the current week's sales to calculate the current week's rolling mean.
weekly_df['rolling_mean_4'] = family_group.transform(lambda x: x.shift(1).rolling(window=4).mean())
weekly_df['rolling_mean_8'] = family_group.transform(lambda x: x.shift(1).rolling(window=8).mean())
weekly_df['rolling_std_4']  = family_group.transform(lambda x: x.shift(1).rolling(window=4).std())

# View the results
weekly_df.head(15)


/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/1723592279.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  weekly_sales = final_train_df.groupby(['family', pd.Grouper(key='date', freq='W-SUN')]).agg(
/var/folders/pz/dvzlv8td7sqcf8m7zn92r2_80000gn/T/ipykernel_69081/1723592279.py:44: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  family_group = weekly_df.groupby('family')['sales']


,family,date,sales,onpromotion,holiday_count,is_holiday_week,max_days_since_earthquake,avg_oil_price,total_week_number,month,...,year,lag_1,lag_2,lag_3,lag_4,lag_12,lag_13,rolling_mean_4,rolling_mean_8,rolling_std_4
0,AUTOMOTIVE,2013-01-06,1287.0,0,6,True,0,93.101667,1,1,...,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AUTOMOTIVE,2013-01-13,1514.0,0,7,True,0,93.442857,2,1,...,2013,1287.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AUTOMOTIVE,2013-01-20,1511.0,0,7,True,0,94.875714,3,1,...,2013,1514.0,1287.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AUTOMOTIVE,2013-01-27,1517.0,0,7,True,0,95.365714,4,1,...,2013,1511.0,1514.0,1287.0,NaN,NaN,NaN,NaN,NaN,NaN
4,AUTOMOTIVE,2013-02-03,1807.0,0,7,True,0,97.368571,5,2,...,2013,1517.0,1511.0,1514.0,1287.0,NaN,NaN,1457.25,NaN,113.526429
5,AUTOMOTIVE,2013-02-10,1940.0,0,7,True,0,96.042857,6,2,...,2013,1807.0,1517.0,1511.0,1514.0,NaN,NaN,1587.25,NaN,146.520476
6,AUTOMOTIVE,2013-02-17,1794.0,0,7,True,0,96.667143,7,2,...,2013,1940.0,1807.0,1517.0,1511.0,NaN,NaN,1693.75,NaN,214.555937
7,AUTOMOTIVE,2013-02-24,1668.0,0,7,True,0,94.244286,8,2,...,2013,1794.0,1940.0,1807.0,1517.0,NaN,NaN,1764.50,NaN,177.701060
8,AUTOMOTIVE,2013-03-03,1673.0,0,7,True,0,91.767143,9,3,...,2013,1668.0,1794.0,1940.0,1807.0,NaN,NaN,1802.25,1629.750,111.188653
9,AUTOMOTIVE,2013-03-10,1520.0,0,7,True,0,91.291429,10,3,...,2013,1673.0,1668.0,1794.0,1940.0,NaN,NaN,1768.75,1678.000,128.170134


In [82]:
weekly_df.columns

Index(['family', 'date', 'sales', 'onpromotion', 'holiday_count',
       'is_holiday_week', 'max_days_since_earthquake', 'avg_oil_price',
       'total_week_number', 'month', 'quarter', 'year', 'lag_1', 'lag_2',
       'lag_3', 'lag_4', 'lag_12', 'lag_13', 'rolling_mean_4',
       'rolling_mean_8', 'rolling_std_4'],
      dtype='object')

In [81]:
weekly_df.describe()

,date,sales,onpromotion,holiday_count,max_days_since_earthquake,avg_oil_price,total_week_number,month,quarter,year,lag_1,lag_2,lag_3,lag_4,lag_12,lag_13,rolling_mean_4,rolling_mean_8,rolling_std_4
count,7986,7.986000e+03,7986.000000,7986.000000,7986.000000,7986.000000,7986.00000,7986.000000,7986.000000,7986.000000,7.953000e+03,7.920000e+03,7.887000e+03,7.854000e+03,7.590000e+03,7.557000e+03,7.854000e+03,7.722000e+03,7854.000000
mean,2015-04-29 12:00:00,1.373121e+05,1001.088655,6.958678,0.326446,67.862973,121.50000,6.214876,2.409091,2014.851240,1.376903e+05,1.375841e+05,1.373470e+05,1.371701e+05,1.351725e+05,1.349891e+05,1.378774e+05,1.380611e+05,15317.492087
min,2013-01-06 00:00:00,0.000000e+00,0.000000,2.000000,0.000000,28.480000,1.00000,1.000000,1.000000,2013.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,2014-03-02 00:00:00,9.220000e+02,0.000000,7.000000,0.000000,46.405714,61.00000,3.000000,1.000000,2014.000000,9.220000e+02,9.157500e+02,9.055000e+02,8.977500e+02,8.390000e+02,8.310000e+02,1.043438e+03,1.247844e+03,109.205339
50%,2015-04-29 12:00:00,7.953672e+03,2.000000,7.000000,0.000000,53.343571,121.50000,6.000000,2.000000,2015.000000,7.984456e+03,7.953672e+03,7.930000e+03,7.924795e+03,7.831123e+03,7.809619e+03,8.266939e+03,8.477580e+03,883.657288
75%,2016-06-26 00:00:00,1.141446e+05,419.750000,7.000000,0.000000,95.840000,182.00000,9.000000,3.000000,2016.000000,1.144920e+05,1.145026e+05,1.144018e+05,1.142173e+05,1.126368e+05,1.126030e+05,1.165157e+05,1.179249e+05,8698.648397
max,2017-08-20 00:00:00,3.204106e+06,44784.000000,7.000000,29.000000,109.185714,242.00000,12.000000,4.000000,2017.000000,3.204106e+06,3.204106e+06,3.204106e+06,3.204106e+06,3.204106e+06,3.204106e+06,2.384207e+06,2.018227e+06,770707.172603
std,NaN,3.212281e+05,3049.726139,0.349678,2.639044,25.631146,69.86316,3.380286,1.099468,1.352531,3.217741e+05,3.216158e+05,3.210876e+05,3.207544e+05,3.161461e+05,3.158023e+05,3.191604e+05,3.177064e+05,46778.438354


In [70]:
segmentation_variables = ['store_nbr', 'family', 'cluster', 'city', 'state', 'type']

segmentation_df = final_train_df[segmentation_variables]
segmentation_df

,store_nbr,family,cluster,city,state,type
0,1,AUTOMOTIVE,13,Quito,Pichincha,D
1,1,BABY CARE,13,Quito,Pichincha,D
2,25,BEAUTY,1,Salinas,Santa Elena,D
3,25,BEVERAGES,1,Salinas,Santa Elena,D
4,1,BOOKS,13,Quito,Pichincha,D
...,...,...,...,...,...,...
55567,3,POULTRY,8,Quito,Pichincha,D
55568,44,PREPARED FOODS,5,Quito,Pichincha,A
55569,3,PRODUCE,8,Quito,Pichincha,D
55570,45,SCHOOL AND OFFICE SUPPLIES,11,Quito,Pichincha,A


In [87]:
segmentation_test_df = final_test_df[segmentation_variables]

# Export datasets

In [89]:
# Export to CSV without the dataframe index
weekly_df.to_csv(os.path.join(DATA, 'weekly_features.csv'), index=False)

inventory_df.to_csv(os.path.join(DATA, 'inventory_features.csv'), index=False)
inventory_test_df.to_csv(os.path.join(DATA, 'inventory_test_features.csv'), index=False)

segmentation_df.to_csv(os.path.join(DATA, 'segmentation_features.csv'), index=False)
segmentation_test_df.to_csv(os.path.join(DATA, 'segmentation_test_features.csv'), index=False)

final_train_df.to_csv(os.path.join(DATA, 'final_train_df.csv'), index=False)
final_test_df.to_csv(os.path.join(DATA, 'final_test_df.csv'), index=False)